# **Building a RAG System with Langchain and ChromaDB**
## **Introduction**
**Retrieval Augmented Generation (RAG)** is a powerful technique that combines the capabilities of large language models with external knowledge
retrieval. This notebook will walk you through building a complete RAG system using:

- **Langchain**: A framework for developing applications powered by language models
- **ChromaDB**: An open-source vector database for storing and retrieving embeddings
- **OpenAI**: For embedding and language model (you can substitute with other providers)

In [75]:
import os

# langchain imports
from dotenv import load_dotenv
from langchain.schema import Document # for providing schema to documents
from langchain.document_loaders import TextLoader # for loading text documents
from langchain.text_splitter import RecursiveCharacterTextSplitter # for splitting loaded documents into chunks
from langchain.embeddings import OpenAIEmbeddings # to embed chunked documents
from langchain.vectorstores import Chroma # to store embedded chunked documents
from langchain_openai import ChatOpenAI

# utility imports
import numpy as np
from typing import List

In [76]:
load_dotenv() # load env variables

True

### **RAG Architecture:**

1. **Document Loading:** Load document from various sources.
2. **Document Splitting:** Break documents into various chunks.
3. **Embedding Generation:** Convert chunks into vector representations.
4. **Vector Storage:** Store embeddings in ChromaDB.
5. **Query Processing:** Convert user query to embedding.
6. **Similarity Search:** Find relevant chunks from vector store.
7. **Context Augmentation:** Combine retrieved chunks with query.
8. **Response Generation:** LLM generates answer using context.

### **Benefits of RAG:**
- Reduces hallucinations.
- Provides up-to-date information.
- Allows citing sources.
- Works with domain-specific knowledge.

### **1. Sample Data**

In [4]:
sample_docs = [
    """
    Neural Networks and Deep Learning

    Neural Networks and Deep Learning are foundational technologies in modern artificial intelligence that enable systems to learn complex patterns from data. At their core, neural networks are composed of layers of interconnected nodes or neurons, each performing a weighted sum of inputs followed by a non-linear activation function. Key activation functions include ReLU, sigmoid, and tanh, and selecting the right activation impacts gradient flow and convergence. Training is typically performed via backpropagation combined with gradient-based optimizers such as SGD, Adam, or RMSProp; learning rate schedules and adaptive optimizers are crucial tuning knobs. Architectures vary by task: convolutional neural networks (CNNs) exploit spatial locality for images, recurrent networks and LSTMs handled sequential data historically, and transformers now dominate many sequence tasks by leveraging self-attention to model relationships regardless of distance. Regularization techniques — dropout, weight decay, batch normalization, and data augmentation — mitigate overfitting and improve generalization. Transfer learning and fine-tuning pretrained models have become standard practice to reduce data and compute needs, enabling practical deployment even when labeled data are scarce. Loss functions must match objectives: cross-entropy for classification, MSE for regression, and specialized losses for ranking or structured outputs. Evaluation metrics like precision, recall, F1, ROC-AUC, and BLEU for generation tasks help quantify performance and trade-offs. Practical deep learning also involves considerations of data quality, labeling strategy, class imbalance, and hardware acceleration (GPUs/TPUs). Efficiency optimizations such as mixed-precision training, model pruning, quantization, and knowledge distillation allow models to run in production under latency and memory constraints. Finally, reproducibility, experiment tracking, and robust pipelines are essential for iterating and deploying reliable neural network solutions.
    """,
    """
    Distributed Systems and Microservices

    Distributed Systems and Microservices design address the needs of scalability, resilience, and maintainability in large-scale software platforms. A distributed system consists of multiple independent nodes that coordinate to provide a unified service; challenges include partial failures, asynchronous communication, and state consistency. Fundamental trade-offs are captured by the CAP theorem which states that a distributed system can only guarantee two of the three: consistency, availability, and partition tolerance. Practical systems use consensus algorithms like Paxos and Raft for leader election and maintaining consistent replicated state across nodes. Data replication strategies (leader-follower, multi-leader, or CRDTs for conflict-free merging) influence latency, throughput, and convergence behavior. Microservices decompose a monolith into cohesive services communicating over lightweight APIs (HTTP/REST, gRPC, or message buses), enabling independent development and deployment but introducing distributed tracing and operational complexity. Patterns such as circuit breakers, bulkheads, retries with exponential backoff, and idempotency handling improve reliability under partial failures. Observability—comprising logs, metrics, and distributed traces—lets teams identify bottlenecks and failures; standardized formats (OpenTelemetry) and tools (Prometheus, Jaeger) are common. Service discovery, API gateways, and sidecar proxies (e.g., Istio, Linkerd) manage traffic routing, security, and telemetry. Data ownership and cross-service transactions often require sagas or compensating transactions rather than distributed two-phase commits to maintain availability. Designing for backward compatibility, versioning, and contract testing ensures safe evolution. Finally, capacity planning, autoscaling policies, and chaos engineering exercises help validate that the distributed system can tolerate real-world failures while meeting performance and reliability goals.
    """,
    """
    Containerization and Kubernetes

    Containerization and Kubernetes have transformed how applications are packaged, deployed, and managed. Containers bundle application code with its runtime, libraries, and dependencies into lightweight, portable images following OCI standards. Docker popularized container workflows with build, tag, and push mechanics; images are layered and immutable, enabling efficient caching and distribution. A container runtime (containerd, CRI-O) executes these images on hosts, providing process isolation via namespaces and cgroups. Kubernetes orchestrates containers across clusters, abstracting compute resources and enabling declarative management. Core Kubernetes primitives include Pods (one or more containers sharing networking and storage), Deployments (declarative updates and scaling), Services (stable network endpoints), ConfigMaps and Secrets for configuration, and PersistentVolumes for durable storage. The control plane components—api-server, scheduler, controller-manager, and etcd—coordinate state and ensure desired state reconciliation. Networking models (CNI plugins) handle pod-to-pod communication, while NetworkPolicies restrict traffic. Ingress controllers manage external HTTP(S) routing, and Service meshes provide observability, security, and traffic control at the sidecar layer. Operational best practices include immutable infrastructure, readiness and liveness probes, resource requests and limits, horizontal pod autoscaling, and using namespaces for multi-tenant isolation. Helm charts and operators automate complex deployments and lifecycle management. Security considerations span image provenance scanning, RBAC, Pod Security Policies (or OPA/Gatekeeper), and runtime hardening. CI/CD pipelines integrate with image registries and Kubernetes to enable automated, safe deployments. Observability through metrics, logging, and tracing, along with chaos testing, ensures reliability and aids rapid troubleshooting in production environments.
    """,
    """
    Data Engineering and ETL Pipelines

    Data Engineering and ETL Pipelines form the backbone of data-driven organizations by enabling ingestion, transformation, storage, and delivery of reliable datasets for analytics and machine learning. Ingestion patterns include batch loads from databases and files, real-time streaming via message brokers (Apache Kafka, Pulsar), and change data capture (CDC) to capture transactional updates. Processing frameworks span batch-oriented systems (Apache Spark, Hive) and stream processors (Flink, Spark Streaming, Kafka Streams) with trade-offs in latency, throughput, and semantics. ETL (Extract, Transform, Load) or ELT workflows handle data cleaning, normalization, enrichment, and joins; schema evolution and type handling are common challenges addressed by schema registry services (Avro, Protobuf, JSON Schema). Data modeling (dimensional models, star schemas, or wide-table approaches) depends on query patterns and analytical needs. Orchestration tools like Apache Airflow, Dagster, or Prefect manage dependencies, scheduling, and retries while providing observability into pipeline runs. Ensuring data quality requires validation checks, anomaly detection, lineage tracking, and unit tests for transformations. Storage choices—data lakes (object stores like S3 with parquet/ORC formats), data warehouses (Snowflake, BigQuery, Redshift), and lakehouses—balance cost, performance, and concurrency. Performance optimizations include partitioning, bucketing, compaction, and appropriate file formats to reduce I/O. Metadata, governance, and access control are enforced through catalogs and policies to ensure compliance and discoverability. Monitoring and alerting for pipeline failures, drift, and SLA violations keep pipelines reliable. Finally, reproducible pipelines, idempotent processing, and clear contracts between producers and consumers are essential for building maintainable, production-grade data platforms that support analytics and ML lifecycles.
    """
]

In [5]:
# save sample documents to files
import tempfile

temp_dir = tempfile.mkdtemp()

for i, doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt", "w") as f:
        f.write(doc)

print(f"Sample documents created in : {temp_dir}")

Sample documents created in : /var/folders/5l/5z3454654d5b9c9jrn1pkgjc0000gn/T/tmpofc4sb24


### **2. Document Loading**

In [6]:
from langchain.document_loaders import DirectoryLoader

loader = DirectoryLoader(
    temp_dir,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
)

documents = loader.load()

print(f"Loaded {len(documents)} documents")
print(f"First document preview: ")
print(documents[0].page_content[:200] + "...")

Loaded 4 documents
First document preview: 

    Data Engineering and ETL Pipelines

    Data Engineering and ETL Pipelines form the backbone of data-driven organizations by enabling ingestion, transformation, storage, and delivery of reliable ...


### **3. Document Splitting**

In [7]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500, # maximum size of each chunk
    chunk_overlap = 50, # overlap between chunk to maintain context
    length_function = len,
    separators=[" "] # hierarchy of separators
)

document_chunks = recursive_splitter.split_documents(documents)

print(f"Created {len(document_chunks)} chunks from {len(documents)} documents")
print(f"\nChunk example: ")
print(f"\nContent: {document_chunks[0].page_content[:150]}...")
print(f"\nMetadata: {document_chunks[0].metadata}")

Created 20 chunks from 4 documents

Chunk example: 

Content: Data Engineering and ETL Pipelines

    Data Engineering and ETL Pipelines form the backbone of data-driven organizations by enabling ingestion, trans...

Metadata: {'source': '/var/folders/5l/5z3454654d5b9c9jrn1pkgjc0000gn/T/tmpofc4sb24/doc_3.txt'}


### **4. Embedding Models**

In [8]:
embeddings = OpenAIEmbeddings(
    base_url=os.getenv("IQ_BASE_URL"),
    model=os.getenv("EMBEDDING_MODEL"),
)

/var/folders/5l/5z3454654d5b9c9jrn1pkgjc0000gn/T/ipykernel_12016/4264793788.py:1: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import OpenAIEmbeddings`.
  embeddings = OpenAIEmbeddings(


### **5. Initialize ChromaDB Vector Store and Stores the chunks in Vector Representation**

In [11]:
# create a chromadb vector store
persist_directory = "./chroma_db"

# initialise chromedb with open ai embeddings
vectorstore = Chroma.from_documents(
    documents=document_chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="rag_collections"
)

print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to : {persist_directory}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector store created with 40 vectors
Persisted to : ./chroma_db


### **6. Test Similarity Search**

In [13]:
query = "what is Neural networks and Microservices"

similar_docs = vectorstore.similarity_search(query, k=3)

print(f"Query: {query}")
print(f"\n Top {len(similar_docs)} similar chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(f"{doc.page_content[:200]}...")
    print(f"Source: {doc.metadata.get('source', 'unknown')}")

Query: what is Neural networks and Microservices

 Top 3 similar chunks:

--- Chunk 1 ---
Neural Networks and Deep Learning

    Neural Networks and Deep Learning are foundational technologies in modern artificial intelligence that enable systems to learn complex patterns from data. At the...
Source: /var/folders/5l/5z3454654d5b9c9jrn1pkgjc0000gn/T/tmp1j9k0e9_/doc_0.txt

--- Chunk 2 ---
Neural Networks and Deep Learning

    Neural Networks and Deep Learning are foundational technologies in modern artificial intelligence that enable systems to learn complex patterns from data. At the...
Source: /var/folders/5l/5z3454654d5b9c9jrn1pkgjc0000gn/T/tmpofc4sb24/doc_0.txt

--- Chunk 3 ---
Distributed Systems and Microservices

    Distributed Systems and Microservices design address the needs of scalability, resilience, and maintainability in large-scale software platforms. A distribut...
Source: /var/folders/5l/5z3454654d5b9c9jrn1pkgjc0000gn/T/tmp1j9k0e9_/doc_1.txt


### **7. Advanced Similarity Search with Scores**

#### **Understanding Similarity Scores**

The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

**ChromaDB default:** Use L2 distance (Euclidean distance)

1. Lower score = MORE similar (closer in vector space)
2. Score of 0 = identical vectors
3. Typical range: 0 to 2 (but can be higher)


**Cosine similarity (if configured):**

1. Higher scores = MORE similar
2. Range: -1 to 1 (1 being identical)

In [14]:
result_vectors = vectorstore.similarity_search_with_score(query, k=3)
print(result_vectors)

[(Document(metadata={'source': '/var/folders/5l/5z3454654d5b9c9jrn1pkgjc0000gn/T/tmp1j9k0e9_/doc_0.txt'}, page_content='Neural Networks and Deep Learning\n\n    Neural Networks and Deep Learning are foundational technologies in modern artificial intelligence that enable systems to learn complex patterns from data. At their core, neural networks are composed of layers of interconnected nodes or neurons, each performing a weighted sum of inputs followed by a non-linear activation function. Key activation functions include ReLU, sigmoid, and tanh, and selecting the right activation impacts gradient flow and'), 0.8715799792060733), (Document(metadata={'source': '/var/folders/5l/5z3454654d5b9c9jrn1pkgjc0000gn/T/tmpofc4sb24/doc_0.txt'}, page_content='Neural Networks and Deep Learning\n\n    Neural Networks and Deep Learning are foundational technologies in modern artificial intelligence that enable systems to learn complex patterns from data. At their core, neural networks are composed of la

### **8. Initialise LLM, RAG Chain, Prompt Template, Query the RAG system**

In [23]:
llm = ChatOpenAI(
    model=os.getenv("CHAT_MODEL"),
    base_url=os.getenv("IQ_BASE_URL")
)

test_response = llm.predict("What is LLM?")
print(test_response)

An **LLM** can refer to two different things depending on context:

1. **Large Language Model**  
   - In artificial intelligence, an LLM is a type of machine learning model trained on vast amounts of text data to understand and generate human language.  
   - Examples include GPT (like the one you're interacting with), PaLM, and LLaMA.  
   - These models are used for tasks such as text completion, translation, summarization, Q&A, and more.  
   - They rely on deep learning techniques, particularly transformer architectures.

2. **Master of Laws (Legum Magister)**  
   - In education, an LLM is a postgraduate academic degree in law.  
   - It’s usually pursued after obtaining a first law degree (like an LLB or JD).  
   - Specializations might include international law, human rights law, tax law, or corporate law.

**Which one are you asking about** — the AI-related meaning or the legal degree?


#### **Modern RAG Chain**

In [31]:
from langchain.chains import create_retrieval_chain
from langchain.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain

In [28]:
# convert vector store to retrieval
retriever = vectorstore.as_retriever(
    search_kwargs={"k":3} # retrieve top 3 relevant chunks
)

print(retriever)

tags=['Chroma', 'OpenAIEmbeddings'] vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x15afad810> search_kwargs={'k': 3}


In [38]:
# create a prompt template
system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Context: {context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

In [44]:
document_chain = create_stuff_documents_chain(llm,prompt)

print(document_chain)

print(
"""
This chain:

    - Takes retrieved documents
    - "Stuffs" them into the prompt's {context} placeholder
    - Sends the complete prompt to the LLM
    - Returns the LLM's response
"""
)


bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), config={'run_name': 'format_inputs'})
| ChatPromptTemplate(input_variables=['context', 'input'], messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse three sentences maximum and keep the answer concise.\n\nContext: {context}")), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}'))])
| ChatOpenAI(client=<openai.resources.chat.completions.Completions object at 0x148ab3810>, async_client=<openai.resources.chat.completions.AsyncCompletions object at 0x12ea0f5d0>, root_client=<openai.OpenAI object at 0x15f5ca050>, root_async_client=<openai.AsyncOpenAI object at 0x148ab0a10>, model_name='gpt-5-chat', openai_api_key=SecretStr(

In [47]:
# create the final rag chain
rag_chain = create_retrieval_chain(retriever, document_chain)

# from the below print you should see chains
# first VectorStoreRetriever which retrieves context from the vector store
# then ChatPromptTemplate - updates the retrieved context in ChatPromptTemplate's context placeholder
# then ChatOpenAI - make a call to our llm mode with the prompt augmented with retrieved context
# then StrOutputParser - output the response from the llm model
print(rag_chain)

bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x15afad810>, search_kwargs={'k': 3}), config={'run_name': 'retrieve_documents'})
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), config={'run_name': 'format_inputs'})
            | ChatPromptTemplate(input_variables=['context', 'input'], messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse three sentences maximum and keep the answer concise.\n\nContext: {context}")), HumanMessagePromptTemplate

In [50]:
response = rag_chain.invoke({"input": "what is neural networks?"})
response

{'input': 'what is neural networks?',
 'context': [Document(metadata={'source': '/var/folders/5l/5z3454654d5b9c9jrn1pkgjc0000gn/T/tmp1j9k0e9_/doc_0.txt'}, page_content='Neural Networks and Deep Learning\n\n    Neural Networks and Deep Learning are foundational technologies in modern artificial intelligence that enable systems to learn complex patterns from data. At their core, neural networks are composed of layers of interconnected nodes or neurons, each performing a weighted sum of inputs followed by a non-linear activation function. Key activation functions include ReLU, sigmoid, and tanh, and selecting the right activation impacts gradient flow and'),
  Document(metadata={'source': '/var/folders/5l/5z3454654d5b9c9jrn1pkgjc0000gn/T/tmpofc4sb24/doc_0.txt'}, page_content='Neural Networks and Deep Learning\n\n    Neural Networks and Deep Learning are foundational technologies in modern artificial intelligence that enable systems to learn complex patterns from data. At their core, neura

In [54]:
response.get("answer")

'Neural networks are computational models inspired by the human brain that learn patterns from data through layers of interconnected nodes (neurons). Each neuron processes inputs using weighted sums and non-linear activation functions like ReLU, sigmoid, or tanh. They are a core technology in deep learning, enabling AI systems to solve complex tasks such as image recognition, language processing, and prediction.'

#### **Create RAG Chain Alternative - Using LCEL (LangChain Expression Language)**

In [56]:
# even more flexible approach using LCEL
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [96]:
custom_prompt = ChatPromptTemplate.from_template(""" Use the following prompt to use the answer.
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support the answer.

Context: {context}

Question: {question}

Answer:""")

# format the output documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# build the chain using LCEL
rag_chain_lcel = (
    {
        "context": retriever | format_docs, 
        "question": RunnablePassthrough()
    }
    | custom_prompt
    | llm
    | StrOutputParser()
)

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x15afad810>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], template=" Use the following prompt to use the answer.\nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support the answer.\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer:"))])
| ChatOpenAI(client=<openai.resources.chat.completions.Completions object at 0x148ab3810>, async_client=<openai.resources.chat.completions.AsyncCompletions object at 0x12ea0f5d0>, root_client=<openai.OpenAI object at 0x15f5ca050>, root_async_client=<openai.AsyncOpenAI object at 0x148ab0a10>, model_name='gpt-5-chat', openai_a

In [97]:
result = rag_chain_lcel.invoke("what is neural networks")
result 

'Neural networks are computational models composed of layers of interconnected nodes, or "neurons," that are designed to learn complex patterns from data. Each neuron performs a weighted sum of its inputs and applies a non-linear activation function, such as ReLU, sigmoid, or tanh, to produce an output. These networks are foundational technologies in modern artificial intelligence and can be trained using techniques like backpropagation along with gradient-based optimizers (e.g., SGD, Adam, RMSProp) to adjust their weights for improved performance. Different architectures, such as convolutional neural networks (CNNs) for images, recurrent networks or LSTMs for sequential data, and transformers for many sequence tasks, adapt neural networks to specific types of problems.'

### **9. Adding New Documents To Existing Vector Store**

In [71]:
# add new documents to existing vector store
new_document = """
Quantum Computing Fundamentals

Quantum computing represents a revolutionary paradigm in computational science, leveraging the principles of quantum mechanics to process information in ways that classical computers cannot. Unlike traditional bits, which are binary and can be either 0 or 1, quantum bits or qubits can exist in multiple states simultaneously due to superposition. This property allows quantum computers to explore vast solution spaces exponentially faster for certain problems, such as factoring large numbers or simulating molecular interactions. Entanglement, another quantum phenomenon, links qubits so that the state of one instantly influences the other, enabling complex correlations that underpin quantum algorithms. Decoherence, however, poses a significant challenge, as environmental interactions can disrupt quantum states, necessitating error correction techniques and cryogenic cooling to maintain coherence.

The history of quantum computing traces back to the 1980s, with Richard Feynman's proposal that quantum systems could simulate other quantum systems more efficiently than classical ones. David Deutsch later formalized the concept of a universal quantum computer, leading to the development of quantum algorithms like Shor's algorithm for integer factorization and Grover's algorithm for database searching. These algorithms demonstrate quantum supremacy in specific domains, where quantum computers can outperform classical counterparts. Hardware advancements have progressed from early proof-of-concept devices to scalable systems, with companies like IBM, Google, and Rigetti building quantum processors using superconducting circuits, trapped ions, or photonic approaches. Quantum gates, analogous to classical logic gates, manipulate qubits through operations like Hadamard gates for superposition or CNOT gates for entanglement.

Practical applications of quantum computing span cryptography, where Shor's algorithm threatens current encryption standards like RSA, prompting the development of quantum-resistant cryptography. In drug discovery, quantum simulations can model molecular structures and reactions at atomic precision, accelerating the design of new pharmaceuticals. Optimization problems in logistics, finance, and supply chain management could benefit from quantum algorithms that evaluate multiple scenarios concurrently. Machine learning is also poised for transformation, with quantum-enhanced algorithms potentially speeding up training and inference in neural networks. However, realizing these benefits requires overcoming technical hurdles, including scaling qubit counts, reducing error rates, and developing robust quantum software ecosystems.

Quantum error correction is crucial for building reliable systems, using codes like the surface code to detect and correct errors without collapsing quantum states. Hybrid quantum-classical approaches, such as variational quantum eigensolvers, combine quantum hardware with classical optimization to tackle near-term problems. The field is interdisciplinary, drawing from physics, computer science, mathematics, and engineering, fostering collaborations across academia and industry. As quantum technology matures, it promises to solve previously intractable problems, potentially reshaping industries from healthcare to climate modeling. Yet, ethical considerations arise, including the impact on cybersecurity and the need for equitable access to quantum resources. Ongoing research aims to democratize quantum computing, making it accessible through cloud-based platforms and open-source tools.

In summary, quantum computing harnesses quantum mechanics to unlock unprecedented computational power, with applications ranging from cryptography to scientific simulations. While challenges like decoherence and scalability persist, rapid progress in algorithms, hardware, and error correction is paving the way for a quantum future. This technology not only advances our understanding of the universe but also drives innovation in solving complex real-world problems.
"""

In [74]:
new_doc = Document(
    page_content=new_document,
    metadata={
        "source": "manual addition", 
        "topic": "quantum computing"
    }
)

In [78]:
# split new document in chunks
new_chunks = recursive_splitter.split_documents([new_doc])

In [ ]:
# add new documents to vector store
vectorstore.add_documents(new_chunks)

['9e046fe7-0c59-4f67-bc28-7cf592d53da5',
 '2f948728-eceb-4074-b808-c3b8d54e74f6',
 '34e06933-3d10-4d1e-87cf-27d4ec40b24b',
 'e4dbb9b0-f550-4619-88a4-dc7e9d0dc22b',
 'f571b7bb-0c1e-49a4-a61d-09ab1205ab73',
 '5e76b713-b833-423a-be40-399fcc029a26',
 '222dad52-ccaa-4326-a403-0ca6868f8d42',
 '115f7b0f-1b3d-4a69-86ca-301b02cbfe40',
 'a5904939-cadd-461e-997a-a3c008faf1c0']

In [90]:
print(f"Added {len(new_chunks)} chunks to the vector store")
print(f"Total vectors now: {vectorstore._collection.count()}")

Added 9 chunks to the vector store
Total vectors now: 49


In [94]:
# query with the updated vector store
new_question = "what is quantum computing as per the context provided?"

response = llm.invoke(new_question)
print(response.content)

Quantum computing is a type of computing that uses the principles of quantum mechanics — the science that explains how things work at the smallest scales, like atoms and subatomic particles.  

Unlike classical computers, which process information using bits that can be either 0 or 1, quantum computers use **quantum bits (qubits)**. Qubits can exist in a **superposition** of states, meaning they can be both 0 and 1 at the same time to varying degrees. They can also be **entangled**, allowing qubits to be linked in ways that make their states dependent on each other, even over long distances.  

This enables quantum computers to perform certain types of calculations much faster and more efficiently than classical computers — especially in fields like cryptography, optimization, simulation of quantum systems, and machine learning. However, quantum computing is still in a relatively early stage of development, with challenges around stability (decoherence), error correction, and scalabili

In [100]:
rag_chain_lcel.invoke(new_question)

'Quantum computing, as per the context provided, is a revolutionary paradigm in computational science that leverages the principles of quantum mechanics to process information in ways that classical computers cannot. Instead of traditional binary bits, quantum computers use quantum bits or qubits, which can exist in multiple states simultaneously due to the property of superposition. This enables quantum computers to explore vast solution spaces exponentially faster for certain problems. The technology has applications ranging from cryptography to scientific simulations, advancing our understanding of the universe and driving innovation in solving complex real-world problems. It originated in the 1980s with Richard Feynman’s proposal for simulating quantum systems efficiently, followed by David Deutsch’s formalization of a universal quantum computer and the creation of powerful quantum algorithms such as Shor’s and Grover’s.'

### **Advanced RAG Techniques - Conversational Memory [Adding previous conversational context]**

- **create_history_aware_retriever** - Makes the retriever understand conversational context
- **MessagesPlaceholder** - Placeholder for chat history in prompts
- **HumanMessage/AIMessage** - Structured message types for conversation history

In [102]:
from langchain.chains import create_history_aware_retriever
from langchain.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

In [103]:
# create a prompt that includes chat history
contextualize_q_system_prompt = """Given a chat history and the latest user question
which might reference context in that chat history, formulate a standalone question
which can be understood without the chat history. Do NOT answer the question, just reformulate
it if needed and otherwise return is as is."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

In [104]:
# create history aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)

history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x15afad810>, search_kwargs={'k': 3}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': typing.List[typing.Union[langchain_core.messages.ai.AIMessage, langchain_core.messages.human.HumanMessage, langchain_core.messages.chat.ChatMessage, langchain_core.messages.system.SystemMessage, langchain_core.messages.function.FunctionMessage, langchain_core.messages.tool.ToolMessage]]}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], template='Given a chat history and the latest user question\nwhich might reference context in that chat history, formulate a standalone question\nwhich can be understood without the chat history. Do NOT answer the q

In [106]:
# NOTE: To show how to hold history context into our previous pipeline, just add MessagesPlaceholder
# create a new document chain with
qa_system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Context: {context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

conversational_rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain
)